# Hướng dẫn truy cập và fix cứng môi trường bài toán Portfolios Management và các tinh chỉnh cấu hình

## Khởi tạo môi trường

In [6]:
## install finrl library
!sudo apt install swig
!pip install git+https://github.com/AI4Finance-Foundation/FinRL.git

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
swig is already the newest version (4.2.0-2ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 92 not upgraded.
  Cloning https://github.com/AI4Finance-Foundation/FinRL.git to /tmp/pip-req-build-sj_7mif3
  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/FinRL.git /tmp/pip-req-build-sj_7mif3
  Resolved https://github.com/AI4Finance-Foundation/FinRL.git to commit 7e98df461cbc7772f16a545a2f13b521e5d0b8f4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/AI4Finance-Foundation/ElegantRL.git to /tmp/pip-install-26k6fjxg/elegantrl_793fff731c45481e92651d4d68629107
  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/ElegantRL.git /tmp/pip-install-26k6fjxg/elegantrl_793fff731c45481e92651d4d68629

In [7]:
## We also need to install quantstats, because the environment uses it to plot graphs
!pip install quantstats

In [8]:
## Hide matplotlib warnings
import warnings
warnings.filterwarnings('ignore')

import logging
logging.getLogger('matplotlib.font_manager').disabled = True

## Lớp khởi tạo môi trường chính của FinRL

---

Notebook này nhằm mục đích cung cấp một ví dụ về việc sử dụng PortfolioOptimizationEnv (hoặc POE) để huấn luyện một mô hình học tăng cường (reinforcement learning) nhằm giải quyết bài toán tối ưu hóa danh mục đầu tư.

Trong tài liệu này, chúng ta sẽ tái hiện một kiến trúc nổi tiếng có tên là EIIE (Ensemble of Identical Independent Evaluators - tập hợp các bộ đánh giá độc lập giống hệt nhau), được giới thiệu trong bài báo sau:

* Zhengyao Jiang, Dixing Xu, & Jinjun Liang. (2017). A Deep Reinforcement Learning Framework for the Financial Portfolio Management Problem. [https://doi.org/10.48550/arXiv.1706.10059](https://doi.org/10.48550/arXiv.1706.10059).

Bạn nên đọc bài báo này để hiểu rõ thuật toán được triển khai trong notebook.

### Lưu ý

Nếu bạn sử dụng môi trường này, hãy cân nhắc trích dẫn bài báo sau (bên cạnh các tài liệu tham khảo về FinRL):

* Caio Costa, & Anna Costa (2023). POE: A General Portfolio Optimization Environment for FinRL. In *Anais do II Brazilian Workshop on Artificial Intelligence in Finance* (tr. 132–143). SBC. [https://doi.org/10.5753/bwaif.2023.231144](https://doi.org/10.5753/bwaif.2023.231144).

```bibtex
@inproceedings{bwaif,
 author = {Caio Costa and Anna Costa},
 title = {POE: A General Portfolio Optimization Environment for FinRL},
 booktitle = {Anais do II Brazilian Workshop on Artificial Intelligence in Finance},
 location = {João Pessoa/PB},
 year = {2023},
 keywords = {},
 issn = {0000-0000},
 pages = {132--143},
 publisher = {SBC},
 address = {Porto Alegre, RS, Brasil},
 doi = {10.5753/bwaif.2023.231144},
 url = {https://sol.sbc.org.br/index.php/bwaif/article/view/24959}
}

```

In [13]:
!pip install torch_geometric

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 11.4 MB/s  0:00:00


In [14]:
import torch

from sklearn.preprocessing import MaxAbsScaler

from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import GroupByScaler
from finrl.meta.env_portfolio_optimization.env_portfolio_optimization import PortfolioOptimizationEnv
from finrl.agents.portfolio_optimization.models import DRLAgent
from finrl.agents.portfolio_optimization.architectures import EIIE

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

## Điều chỉnh các đầu vào custom có thể diễn ra

### - Đầu vào chuẩn không áp dụng cho phân tích cảm xúc thị trường (Sentiment Analysis)


In [15]:
# ====================================================================
# ÁP DỤNG TRỰC TIẾP MÔI TRƯỜNG PortfolioOptimizationEnv CHÍNH THỨC CỦA FinRL
# (Costa & Costa 2023, BWAIF - Kế thừa trực tiếp, KHÔNG rewrite lại môi trường)
# ====================================================================

class PortfolioOptimizationEnv_v0(PortfolioOptimizationEnv):
    """
    Kế thừa và áp dụng trực tiếp PortfolioOptimizationEnv chính thức của FinRL (Costa & Costa, 2023)
    đã được công bố và kiểm chứng trong file FinRL_PortfolioOptimizationEnv_Demo.ipynb.
    
    Môi trường mô phỏng đầy đủ:
    - Vector biến động giá tương đối y_t = [1.0, p_t / p_{t-1}]
    - Mô hình phí bảo toàn vốn TRF (Transaction Remainder Factor, Jiang et al. 2017)
    - Trôi dạt trọng số thị trường (Price Drift)
    - Phần thưởng log-return: r_t = ln(mu_t * y_t^T * w_t)
    - Không gian quan sát Tensor 3D (F x N x M) và Action Simplex (N + 1)
    """
    def __init__(
        self,
        df: Optional[pd.DataFrame] = None,
        initial_amount: float = 100_000.0,
        comission_fee_pct: float = 0.0025,
        time_window: int = 30,
        features: Optional[List[str]] = None,
        normalize_df: Optional[str] = None,
        return_last_action: bool = False,
        new_gym_api: bool = True,
        # Các tham số hỗ trợ linh hoạt cho cả DataFrame FinRL và ma trận giá
        df_features: Optional[np.ndarray] = None,
        df_prices: Optional[np.ndarray] = None,
        tic_list: Optional[List[str]] = None,
        fee_rate: Optional[float] = None,
        fee_model: str = "trf",
        reward_scaling: float = 1.0,
        **kwargs
    ):
        if features is None:
            features = ["close", "high", "low"]
        if fee_rate is not None:
            comission_fee_pct = fee_rate

        # Chuyển đổi dữ liệu sang định dạng DataFrame chuẩn FinRL nếu truyền ma trận giá
        if df is None and df_prices is not None:
            T_len, N_len = df_prices.shape
            if tic_list is None:
                tic_list = [f"TIC_{i}" for i in range(N_len)]
            date_range = pd.date_range("2020-01-01", periods=T_len, freq="B").strftime("%Y-%m-%d")
            records = []
            for t_idx, d_str in enumerate(date_range):
                for n_idx, tic in enumerate(tic_list):
                    close_p = float(df_prices[t_idx, n_idx])
                    records.append({
                        "date": d_str,
                        "tic": tic,
                        "close": close_p,
                        "high": close_p * 1.002,
                        "low": close_p * 0.998
                    })
            df = pd.DataFrame(records)

        # Gọi trực tiếp khởi tạo của PortfolioOptimizationEnv từ FinRL (KHÔNG rewrite)
        super().__init__(
            df=df,
            initial_amount=initial_amount,
            return_last_action=return_last_action,
            normalize_df=normalize_df,
            reward_scaling=reward_scaling,
            comission_fee_model=fee_model,
            comission_fee_pct=comission_fee_pct,
            features=features,
            time_window=time_window,
            new_gym_api=new_gym_api,
            **kwargs
        )

    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):
        """Khởi tạo lại môi trường tương thích Gymnasium API mới."""
        if seed is not None:
            self._seed(seed)
        res = super().reset()
        if not self._new_gym_api and not isinstance(res, tuple):
            return res, {}
        return res

    def step(self, actions: np.ndarray):
        """Thực thi một bước mô phỏng với PortfolioOptimizationEnv chính thức."""
        res = super().step(actions)
        if len(res) == 5:
            obs, reward, terminal, truncated, info = res
        else:
            obs, reward, terminal, info = res
            truncated = False
        # Bổ sung các trường tiện ích cho tương thích đánh giá trong notebook
        info["fee_factor"] = info.get("trf_mu", 1.0)
        info["portfolio_value"] = self._portfolio_value
        return obs, reward, terminal, truncated, info

    @property
    def action_dim(self) -> int:
        return self.action_space.shape[0]

    @property
    def num_stocks(self) -> int:
        return self.portfolio_size

    @property
    def time_window(self) -> int:
        return self._time_window

    @property
    def num_features(self) -> int:
        return len(self._features)

    @property
    def T(self) -> int:
        return self.episode_length

    @property
    def asset_memory(self) -> list:
        return self._asset_memory.get("final", [self._initial_amount])

    def compute_metrics(self) -> Dict[str, float]:
        """Tính toán các chỉ số tài chính đánh giá hiệu suất danh mục."""
        values = np.array(self.asset_memory)
        if len(values) < 2:
            return {"Final Value ($)": self._portfolio_value}
        returns = np.diff(values) / values[:-1]
        total_return = (values[-1] - values[0]) / values[0]
        ann_return = ((1.0 + total_return) ** (252.0 / len(values))) - 1.0 if len(values) > 0 else 0.0
        ann_volatility = np.std(returns) * np.sqrt(252.0) if len(returns) > 1 else 0.0
        sharpe = (ann_return - 0.02) / (ann_volatility + 1e-8) if ann_volatility > 0 else 0.0
        cummax = np.maximum.accumulate(values)
        drawdowns = (cummax - values) / cummax
        max_drawdown = np.max(drawdowns) if len(drawdowns) > 0 else 0.0
        return {
            'Initial Value ($)': self._initial_amount,
            'Final Value ($)': values[-1],
            'Cumulative Return (%)': total_return * 100.0,
            'Annualized Return (%)': ann_return * 100.0,
            'Annualized Volatility (%)': ann_volatility * 100.0,
            'Sharpe Ratio': sharpe,
            'Max Drawdown (%)': max_drawdown * 100.0
        }

    def plot_results(self, title: str = "Portfolio Value Over Time"):
        """Hiển thị đồ thị tăng trưởng giá trị danh mục."""
        values = self.asset_memory
        plt.figure(figsize=(10, 5))
        plt.plot(values, label="Portfolio Value", color="#1f77b4", lw=2.0)
        plt.title(title, fontsize=12, fontweight="bold")
        plt.xlabel("Trading Steps", fontsize=10)
        plt.ylabel("Portfolio Value ($)", fontsize=10)
        plt.grid(True, alpha=0.3)
        plt.legend(fontsize=10)
        plt.show()

print('✅ Đã áp dụng thành công lớp môi trường PortfolioOptimizationEnv chính thức của FinRL!')


✅ Đã áp dụng thành công lớp môi trường PortfolioOptimizationEnv chính thức của FinRL!
